# POMDP Oil Futures Trading

This notebook implements a POMDP (Partially Observable Markov Decision Process) approach to trading WTI crude oil futures based on weather data.

## Key Components
- **Hidden States**: Market regimes (bull/bear/volatile/calm) learned from historical returns
- **Observations**: Weather features (temperature, precipitation, hurricanes)
- **Actions**: SHORT (-1), FLAT (0), LONG (+1)
- **Belief State**: Probability distribution over hidden states, updated via Bayesian inference

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pomdp_model import WTIPOMDPModel

## 1. Load Data with Hurricane Features

In [ ]:
# Load the merged dataset (includes weather + hurricane data)
df = pd.read_csv('wti_1_data_with_hurricanes.csv')
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nColumns:")
print(df.columns.tolist())

In [ ]:
# Define features and target
feature_cols = [c for c in df.columns if c not in ['Date', 'ret_CL1', 'spot_price', 'CL1']]
return_col = 'ret_CL1'

print(f"Number of features: {len(feature_cols)}")
print(f"\nFeature types:")
print(f"  - Pipeline weather (temp + precip): 16 features")
print(f"  - Producing region weather: 26 features")
print(f"  - Hurricane data: 16 features")

In [ ]:
# Check for NaN values
nan_counts = df.isna().sum()
print(f"NaN values in ret_CL1: {nan_counts['ret_CL1']}")
print("(These will be automatically dropped during model fitting)")

## 2. Train/Test Split

In [ ]:
# Time-based split (80% train, 20% test)
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

print(f"Training data: {len(train_df)} samples ({train_df['Date'].min()} to {train_df['Date'].max()})")
print(f"Test data: {len(test_df)} samples ({test_df['Date'].min()} to {test_df['Date'].max()})")

## 3. POMDP Model

### How it works:
1. **Learn Hidden States**: Cluster historical returns into market regimes using K-means
2. **Learn Transition Model**: P(s' | s) - how market regimes transition
3. **Learn Observation Model**: P(o | s) - how weather relates to market regimes
4. **Belief Updates**: Use Bayesian inference to update belief state after each observation
5. **Q-Learning**: Learn value function over belief states

In [ ]:
# Create and fit POMDP model
pomdp = WTIPOMDPModel(
    n_hidden_states=4,   # Number of market regimes
    n_obs_clusters=8,    # Number of weather observation clusters
    gamma=0.95           # Discount factor
)

pomdp.fit(train_df, feature_cols, return_col)

In [ ]:
# Visualize the learned market regimes
train_returns = train_df.dropna(subset=[return_col])[return_col].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Return distribution by state
for s in range(pomdp.n_states):
    state_returns = train_returns[pomdp.states == s]
    axes[0].hist(state_returns, bins=50, alpha=0.5, label=f'State {s}')

axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Return Distribution by Market Regime')
axes[0].legend()

# Reward matrix
im = axes[1].imshow(pomdp.R, cmap='RdYlGn', aspect='auto')
axes[1].set_xlabel('Action (0=SHORT, 1=FLAT, 2=LONG)')
axes[1].set_ylabel('Market State')
axes[1].set_title('Expected Reward R(s, a)')
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(['SHORT', 'FLAT', 'LONG'])
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 4. Train Q-Learning over Belief States

In [ ]:
# Train Q-learning (belief-based)
training_rewards = pomdp.train_qlearning(
    train_df, 
    feature_cols, 
    return_col,
    n_episodes=100,
    alpha=0.1,
    epsilon_start=1.0,
    epsilon_end=0.1
)

In [ ]:
# Plot training progress
plt.figure(figsize=(10, 4))
plt.plot(training_rewards)
plt.xlabel('Episode')
plt.ylabel('Total Episode Reward')
plt.title('Q-Learning Training Progress')
plt.grid(True, alpha=0.3)
plt.show()

# Rolling average
window = 10
rolling_avg = pd.Series(training_rewards).rolling(window).mean()
print(f"Final {window}-episode average reward: {rolling_avg.iloc[-1]:.4f}")

## 5. Evaluate on Test Set

In [ ]:
# Evaluate POMDP policy
results = pomdp.evaluate(test_df, feature_cols, return_col, use_qlearning=True)

print("=== POMDP Policy Results ===")
print(f"Total Reward: {results['total_reward']:.4f}")
print(f"Mean Daily Reward: {results['mean_reward']:.6f}")
print(f"Sharpe Ratio (annualized): {results['sharpe']:.4f}")

In [ ]:
# Compare to baselines
test_returns = test_df.dropna(subset=[return_col])[return_col].values[:-1]

# Buy and Hold
buy_hold_reward = test_returns.sum()
buy_hold_sharpe = test_returns.mean() / (test_returns.std() + 1e-8) * np.sqrt(252)

# Sell and Hold (short)
sell_hold_reward = -test_returns.sum()

# Random
np.random.seed(42)
random_positions = np.random.choice([-1, 0, 1], size=len(test_returns))
random_reward = (random_positions * test_returns).sum()

print("\n=== Baseline Comparisons ===")
print(f"Buy & Hold:  Total={buy_hold_reward:.4f}, Sharpe={buy_hold_sharpe:.4f}")
print(f"Sell & Hold: Total={sell_hold_reward:.4f}")
print(f"Random:      Total={random_reward:.4f}")
print(f"POMDP:       Total={results['total_reward']:.4f}, Sharpe={results['sharpe']:.4f}")

In [ ]:
# Plot cumulative returns
pomdp_cumulative = np.cumsum(results['rewards'])
buyhold_cumulative = np.cumsum(test_returns)

plt.figure(figsize=(12, 5))
plt.plot(pomdp_cumulative, label='POMDP Strategy', linewidth=2)
plt.plot(buyhold_cumulative, label='Buy & Hold', linewidth=2, alpha=0.7)
plt.xlabel('Trading Day')
plt.ylabel('Cumulative Return')
plt.title('POMDP Strategy vs Buy & Hold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Position distribution
positions = np.array(results['positions'])
position_counts = pd.Series(positions).value_counts().sort_index()

plt.figure(figsize=(8, 4))
plt.bar(['SHORT (-1)', 'FLAT (0)', 'LONG (+1)'], 
        [position_counts.get(-1, 0), position_counts.get(0, 0), position_counts.get(1, 0)])
plt.ylabel('Number of Days')
plt.title('Position Distribution')
plt.show()

print(f"Position breakdown:")
print(f"  SHORT: {position_counts.get(-1, 0)} days ({100*position_counts.get(-1, 0)/len(positions):.1f}%)")
print(f"  FLAT:  {position_counts.get(0, 0)} days ({100*position_counts.get(0, 0)/len(positions):.1f}%)")
print(f"  LONG:  {position_counts.get(1, 0)} days ({100*position_counts.get(1, 0)/len(positions):.1f}%)")

## 6. Hyperparameter Tuning

In [ ]:
# Try different numbers of hidden states
results_by_states = {}

for n_states in [2, 3, 4, 5, 6]:
    print(f"\nTrying n_hidden_states={n_states}...")
    model = WTIPOMDPModel(n_hidden_states=n_states, n_obs_clusters=8)
    model.fit(train_df, feature_cols, return_col)
    model.train_qlearning(train_df, feature_cols, return_col, n_episodes=50)
    result = model.evaluate(test_df, feature_cols, return_col)
    results_by_states[n_states] = result['total_reward']
    print(f"  Total Reward: {result['total_reward']:.4f}")

best_n_states = max(results_by_states, key=results_by_states.get)
print(f"\nBest n_hidden_states: {best_n_states} (reward: {results_by_states[best_n_states]:.4f})")

## 7. Using the Gym Environment

For DQN or other deep RL methods, use the WTIEnv gym environment:

In [ ]:
from wti_env import WTIEnv

# Create environment
env = WTIEnv(
    df=df,
    feature_cols=feature_cols,
    return_col=return_col,
    window=20,
    cost=0.0005,
    normalize=True  # Normalizes features for better RL training
)

print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.n}")

In [ ]:
# Test a random policy
obs, info = env.reset()
total_reward = 0
n_steps = 0

terminated = truncated = False
while not (terminated or truncated):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    n_steps += 1

print(f"Random policy: {n_steps} steps, total reward = {total_reward:.4f}")

## Summary

This notebook demonstrates:
1. **Data preparation**: Merged weather + hurricane data, handled NaN values
2. **POMDP formulation**: Hidden market states, weather observations, trading actions
3. **Belief-based Q-learning**: Learning a policy over belief states
4. **Evaluation**: Comparison with baselines (buy & hold, random)

### Next Steps
- Try more sophisticated belief representations
- Use deep RL (DQN, PPO) with the gymnasium environment
- Add more features (volatility index OVX, technical indicators)
- Explore different reward functions (risk-adjusted returns)